In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
from tqdm import tqdm

%matplotlib inline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('running on', device)

SAVE_FIGURES = False  # set to True to write the figures used on the slides to ./figures
def savefig(name):
    if SAVE_FIGURES:
        os.makedirs('figures', exist_ok=True)
        plt.savefig(f'figures/{name}.png', dpi=150, bbox_inches='tight')

In [ ]:
%%sh
# Download the data - you need to do this only once
wget --no-verbose --output-document=image.jpg https://github.com/chrirupp/cv_course/raw/main/data/image.jpg
wget --no-verbose --output-document=dog_cutout.png https://github.com/chrirupp/cv_course/raw/main/data/dog_cutout.png

# Lecture 13 - Tracking

Lucas-Kanade from scratch, the aperture problem, and point tracking over a whole video: frame-to-frame KLT versus CoTracker3, which tracks many points jointly and reasons about occlusions.

We use a **synthetic** clip (the same as in Lecture 12) so that we know the true trajectory of every point, including when it is hidden.

Run this notebook on Google Colab (see the [readme](https://github.com/chrirupp/cv_course)). Cells that load large pretrained models are faster with a GPU runtime (*Edit > Notebook settings > T4 GPU*).

In [ ]:
# A synthetic video with exact ground truth: a camera pans and zooms over a photo (the background),
# a dog walks through the scene and passes behind a pole that stands in the scene.
# Because we render it ourselves, we know where every pixel goes (flow) and when it is hidden (occlusion).
background = cv2.cvtColor(cv2.imread('image.jpg'), cv2.COLOR_BGR2RGB).astype(np.float32)
cutout = cv2.imread('dog_cutout.png', cv2.IMREAD_UNCHANGED)
cutout = cv2.resize(cutout, (0, 0), fx=0.75, fy=0.75, interpolation=cv2.INTER_AREA)
dog_rgb = cv2.cvtColor(cutout[:, :, :3], cv2.COLOR_BGR2RGB).astype(np.float32)
dog_alpha = cutout[:, :, 3].astype(np.float32) / 255

H, W, T = 360, 480, 48
POLE_X, POLE_WIDTH = 640, 36            # the pole, in background (world) coordinates

def camera(t):
    """scale and offset: background point X appears at s * (X - o) in frame t"""
    return 0.5 * (1 + 0.003 * t), np.array([40 + 5.0 * t, 60 + 1.0 * t])

def dog_position(t):
    """top-left corner of the dog in frame t (image coordinates)"""
    return np.array([20 + 6.0 * t, 120 + 8 * np.sin(t / 5)])

def layers(t):
    """the masks of the dog and the pole in frame t (the pole is in front of the dog)"""
    s, o = camera(t)
    px, py = dog_position(t)
    shift = np.float32([[1, 0, px], [0, 1, py]])
    dog_mask = cv2.warpAffine(dog_alpha, shift, (W, H))
    pole_mask = np.zeros((H, W), np.float32)
    x0, x1 = s * (POLE_X - o[0]), s * (POLE_X + POLE_WIDTH - o[0])
    pole_mask[:, max(int(round(x0)), 0):max(int(round(x1)), 0)] = 1
    return dog_mask, pole_mask, shift

def render(t):
    s, o = camera(t)
    A = np.float32([[s, 0, -s * o[0]], [0, s, -s * o[1]]])
    frame = cv2.warpAffine(background, A, (W, H), flags=cv2.INTER_LINEAR)
    dog_mask, pole_mask, shift = layers(t)
    dog = cv2.warpAffine(dog_rgb, shift, (W, H))
    frame = frame * (1 - dog_mask[..., None]) + dog * dog_mask[..., None]
    frame = frame * (1 - pole_mask[..., None]) + np.array([70, 60, 55]) * pole_mask[..., None]
    return np.clip(frame, 0, 255).astype(np.uint8)

def ground_truth_tracks(points, t0=0):
    """positions (T, N, 2) and visibility (T, N) of points given in frame t0"""
    points = np.asarray(points, np.float32)
    dog_mask, pole_mask, _ = layers(t0)
    xi = np.clip(points[:, 0].round().astype(int), 0, W - 1)
    yi = np.clip(points[:, 1].round().astype(int), 0, H - 1)
    on_dog = dog_mask[yi, xi] > 0.5
    s0, o0 = camera(t0)
    world = points / s0 + o0                    # background points in world coordinates
    tracks, visible = np.zeros((T, len(points), 2), np.float32), np.zeros((T, len(points)), bool)
    for t in range(T):
        s, o = camera(t)
        tracks[t] = np.where(on_dog[:, None], points - dog_position(t0) + dog_position(t), s * (world - o))
        dog_mask, pole_mask, _ = layers(t)
        x, y = tracks[t, :, 0], tracks[t, :, 1]
        inside = (x >= 0) & (x <= W - 1) & (y >= 0) & (y <= H - 1)
        xi, yi = np.clip(x.round().astype(int), 0, W - 1), np.clip(y.round().astype(int), 0, H - 1)
        hidden = (pole_mask[yi, xi] > 0.5) | (~on_dog & (dog_mask[yi, xi] > 0.5))
        visible[t] = inside & ~hidden
    return tracks, visible

def ground_truth_flow(t):
    """dense optical flow from frame t to frame t+1 (H, W, 2)"""
    ys, xs = np.mgrid[0:H, 0:W].astype(np.float32)
    points = np.stack([xs.ravel(), ys.ravel()], axis=1)
    s0, o0 = camera(t)
    s1, o1 = camera(t + 1)
    flow = s1 * (points / s0 + o0 - o1) - points
    dog_mask, pole_mask, _ = layers(t)
    flow[dog_mask.ravel() > 0.5] = dog_position(t + 1) - dog_position(t)
    flow[pole_mask.ravel() > 0.5] = s1 * (points[pole_mask.ravel() > 0.5] / s0 + o0 - o1) - points[pole_mask.ravel() > 0.5]
    return flow.reshape(H, W, 2).astype(np.float32)

video = np.stack([render(t) for t in range(T)])
print('video:', video.shape)

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def animate(frames, interval=80):
    fig, ax = plt.subplots(figsize=(6, 4.5))
    im = ax.imshow(frames[0])
    ax.axis('off')
    anim = FuncAnimation(fig, lambda t: [im.set_data(frames[t])], frames=len(frames), interval=interval)
    plt.close()
    return HTML(anim.to_jshtml())

animate(video)

## Lucas-Kanade for a translation

We want the displacement $\mathbf{d}$ that aligns a template $T$ (a patch of frame 0) with the next frame $I$:
$$E(\mathbf{d}) = \sum_{\mathbf{x}} \left[ I(\mathbf{x} + \mathbf{d}) - T(\mathbf{x}) \right]^2 .$$
Linearising $I$ around the current estimate gives the Gauss-Newton update
$$\Delta \mathbf{d} = - \Big(\sum \nabla I \nabla I^\top\Big)^{-1} \sum \nabla I \left[ I(\mathbf{x} + \mathbf{d}) - T(\mathbf{x}) \right],$$
which we iterate. The linearisation is only valid for small displacements.

In [ ]:
gray = np.stack([cv2.GaussianBlur(cv2.cvtColor(f, cv2.COLOR_RGB2GRAY).astype(np.float32), (0, 0), 1.0) for f in video])

def patch(image, center, size):
    return cv2.getRectSubPix(image, (size, size), (float(center[0]), float(center[1])))

def lucas_kanade(I0, I1, p, size=31, d=(0, 0), iterations=30):
    """estimate the translation of the patch around p from I0 to I1; returns all iterates"""
    Iy, Ix = np.gradient(I1)
    T = patch(I0, p, size)
    d = np.array(d, np.float64)
    history = [d.copy()]
    for _ in range(iterations):
        q = p + d
        error = patch(I1, q, size) - T
        gx, gy = patch(Ix, q, size).ravel(), patch(Iy, q, size).ravel()
        G = np.array([[gx @ gx, gx @ gy], [gx @ gy, gy @ gy]])   # structure tensor
        b = np.array([gx @ error.ravel(), gy @ error.ravel()])
        d = d - np.linalg.solve(G, b)
        history.append(d.copy())
    return np.array(history)

def lucas_kanade_pyramid(I0, I1, p, size=31, levels=3, iterations=30):
    """coarse-to-fine: estimate at low resolution, double, refine at the next level"""
    d = np.zeros(2)
    for level in reversed(range(levels)):
        s = 2 ** level
        small0 = cv2.resize(I0, (0, 0), fx=1/s, fy=1/s, interpolation=cv2.INTER_AREA)
        small1 = cv2.resize(I1, (0, 0), fx=1/s, fy=1/s, interpolation=cv2.INTER_AREA)
        d = lucas_kanade(small0, small1, p / s, size, d / s, iterations)[-1] * s
    return d

# a well-textured background point (the strongest corner in a region far from the dog)
min_eig = cv2.cornerMinEigenVal(gray[0], blockSize=15, ksize=3)
region = np.zeros_like(min_eig, bool)
region[100:300, 340:440] = True
y, x = np.unravel_index(np.argmax(np.where(region, min_eig, 0)), min_eig.shape)
p = np.array([x, y], np.float64)
fig, axs = plt.subplots(1, 3, figsize=(20, 5))
axs[0].imshow(video[0])
axs[0].add_patch(plt.Rectangle(p - 15, 31, 31, fill=False, color='red', linewidth=2))
axs[0].set_title('Template in frame 0')
axs[0].axis('off')
for t, ax in zip([2, 10], axs[1:]):
    true = ground_truth_tracks([p])[0][t, 0] - p
    history = lucas_kanade(gray[0], gray[t], p)
    ax.plot(np.linalg.norm(history - true, axis=1), 'o-', label='Lucas-Kanade')
    ax.axhline(np.linalg.norm(lucas_kanade_pyramid(gray[0], gray[t], p) - true), color='green', linestyle='--', label='coarse-to-fine (3 levels), final')
    ax.set_xlabel('iteration')
    ax.set_ylabel('error of the displacement (px)')
    ax.set_yscale('log')
    ax.set_title(f'Frame 0 -> {t}: true displacement ({true[0]:.1f}, {true[1]:.1f}) px')
    ax.legend()
    ax.grid(alpha=0.3)
savefig('tracking_lucas_kanade')
plt.show()

## The aperture problem

The update needs the inverse of the structure tensor $G = \sum \nabla I \nabla I^\top$. Its eigenvalues tell us whether a patch can be tracked: in a flat region both are small, on an edge only one is large (motion *along* the edge is invisible), at a corner both are large. Shi-Tomasi's "good features to track" are the points where the smaller eigenvalue is large.

In [ ]:
# the larger eigenvalue, to find a flat patch (both eigenvalues small)
max_eig = cv2.cornerEigenValsAndVecs(gray[0], blockSize=15, ksize=3)[:, :, 0]
inner = np.zeros_like(max_eig, bool)
inner[20:-20, 20:280] = True
y, x = np.unravel_index(np.argmin(np.where(inner, max_eig, np.inf)), max_eig.shape)
s, o = camera(0)
pole_left = s * (POLE_X - o[0])
examples = {'flat': np.array([x, y], np.float64), 'edge (of the pole)': np.array([pole_left, 60.0]), 'corner': p}

fig, axs = plt.subplots(1, 4, figsize=(24, 5), gridspec_kw={'width_ratios': [2, 1, 1, 1]})
axs[0].imshow(np.log(min_eig + 1e-3), cmap='magma')
axs[0].set_title('log of the smaller eigenvalue of the structure tensor')
for ax, (name, q) in zip(axs[1:], examples.items()):
    Iy, Ix = np.gradient(gray[0])
    gx, gy = patch(Ix, q, 31).ravel(), patch(Iy, q, 31).ravel()
    eig = np.linalg.eigvalsh(np.array([[gx @ gx, gx @ gy], [gx @ gy, gy @ gy]]))
    ax.imshow(patch(video[0].astype(np.float32), q, 31).astype(np.uint8))
    ax.set_title(f'{name}: eigenvalues {eig[0]:.0f}, {eig[1]:.0f}')
    axs[0].add_patch(plt.Rectangle(q - 15, 31, 31, fill=False, color='cyan', linewidth=2))
for ax in axs:
    ax.axis('off')
savefig('tracking_aperture')
plt.show()

## Tracking points through the whole video

We pick points on the background and on the dog in frame 0 and follow them through all 48 frames.

* **KLT**: pyramidal Lucas-Kanade from frame to frame (OpenCV). Every step adds a little error (drift), and it has no notion of occlusion: when the dog walks behind the pole, its points get stuck on the pole or are lost.
* **CoTracker3**: a transformer that tracks all points *jointly* over a window of frames, using correlation features. It predicts both positions and visibility, and keeps tracking points while they are hidden.

In [ ]:
dog_mask, pole_mask, _ = layers(0)
corners = lambda mask, n: cv2.goodFeaturesToTrack(gray[0], n, 0.01, 8, mask=mask.astype(np.uint8)).reshape(-1, 2)
queries = np.concatenate([corners((dog_mask > 0.9) & (pole_mask < 0.5), 40),
                          corners((dog_mask < 0.1) & (pole_mask < 0.5), 100)])
on_dog = dog_mask[queries[:, 1].astype(int), queries[:, 0].astype(int)] > 0.5
true_tracks, true_visible = ground_truth_tracks(queries)
print(f'{len(queries)} query points, {on_dog.sum()} on the dog')

# KLT: chain pyramidal LK from frame to frame; a point is lost once its status is 0
klt = np.zeros_like(true_tracks)
klt_alive = np.zeros(true_visible.shape, bool)
klt[0], klt_alive[0] = queries, True
for t in range(1, T):
    nxt, status, _ = cv2.calcOpticalFlowPyrLK(video[t - 1], video[t], klt[t - 1].reshape(-1, 1, 2), None, winSize=(21, 21), maxLevel=3)
    klt[t] = nxt.reshape(-1, 2)
    klt_alive[t] = klt_alive[t - 1] & (status.ravel() == 1)
print(f'KLT lost {(~klt_alive[-1]).sum()} of {len(queries)} points')

In [ ]:
cotracker = torch.hub.load('facebookresearch/co-tracker', 'cotracker3_offline').to(device).eval()
video_tensor = torch.from_numpy(video).permute(0, 3, 1, 2)[None].float().to(device)        # (1, T, 3, H, W) in [0, 255]
query_tensor = torch.from_numpy(np.concatenate([np.zeros((len(queries), 1)), queries], axis=1)).float()[None].to(device)  # (t, x, y)
with torch.no_grad():
    tracks, visibility = cotracker(video_tensor, queries=query_tensor)
cotracker_tracks = tracks[0].cpu().numpy()
cotracker_visible = visibility[0].cpu().numpy()
print('occlusion prediction accuracy:', (cotracker_visible == true_visible).mean().round(3))

In [ ]:
def accuracy(tracks, valid, threshold=3):
    """per frame: fraction of all query points that are tracked to within `threshold` px of the truth
    (positions are scored also while a point is hidden - a tracker should know where it is)"""
    inside = (true_tracks[..., 0] >= 0) & (true_tracks[..., 0] < W)
    good = (np.linalg.norm(tracks - true_tracks, axis=2) < threshold) & valid
    return [(good[t][inside[t]]).mean() for t in range(T)], good

klt_acc, klt_good = accuracy(klt, klt_alive)
co_acc, co_good = accuracy(cotracker_tracks, np.ones_like(true_visible))
fig, axs = plt.subplots(1, 3, figsize=(24, 6), gridspec_kw={'width_ratios': [1.2, 1, 1]})
for name, acc, good in [('KLT', klt_acc, klt_good), ('CoTracker3', co_acc, co_good)]:
    axs[0].plot(acc, label=f'{name}: all points')
    axs[0].plot([good[t][on_dog].mean() for t in range(T)], '--', label=f'{name}: points on the dog')
hidden = ~true_visible[:, on_dog].all(axis=1)
axs[0].fill_between(range(T), 0, 1.02, where=hidden, color='gray', alpha=0.2, label='some dog points hidden')
axs[0].set_xlabel('frame')
axs[0].set_ylabel('fraction of points within 3 px')
axs[0].set_ylim(0, 1.02)
axs[0].legend(loc='lower left')
axs[0].grid(alpha=0.3)

t = 40
for ax, (name, tracks, valid, vis) in zip(axs[1:], [('KLT', klt, klt_alive, klt_alive), ('CoTracker3', cotracker_tracks, np.ones_like(true_visible), cotracker_visible)]):
    ax.imshow(video[t])
    for i in np.where(on_dog)[0]:
        if valid[t, i]:
            ax.plot(tracks[:t + 1, i, 0], tracks[:t + 1, i, 1], '-', color='yellow', linewidth=0.8, alpha=0.7)
    ok = np.linalg.norm(tracks[t] - true_tracks[t], axis=1) < 3
    sel = on_dog & valid[t]
    ax.scatter(*tracks[t, sel & ok].T, c='lime', s=25, label='within 3 px')
    ax.scatter(*tracks[t, sel & ~ok].T, c='red', s=25, label='wrong')
    if name == 'CoTracker3':
        ax.scatter(*tracks[t, sel & ~vis[t]].T, facecolors='none', edgecolors='white', s=80, label='predicted hidden')
    ax.set_title(f'{name}, frame {t}: points on the dog ({(~valid[t] & on_dog).sum()} lost)')
    ax.legend(loc='lower right')
    ax.axis('off')
savefig('tracking_klt_vs_cotracker')
plt.show()

In [ ]:
# watch the dog points: KLT (left) and CoTracker3 (right); hollow circles = predicted hidden
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8), dpi=72)
ims = [ax.imshow(video[0]) for ax in axs]
dots = [axs[0].scatter([], [], c='yellow', s=12), axs[1].scatter([], [], c='yellow', s=12), axs[1].scatter([], [], facecolors='none', edgecolors='white', s=40)]
for ax, name in zip(axs, ['KLT', 'CoTracker3']):
    ax.set_title(name)
    ax.axis('off')
def update(t):
    for im in ims:
        im.set_data(video[t])
    dots[0].set_offsets(klt[t, on_dog & klt_alive[t]])
    dots[1].set_offsets(cotracker_tracks[t, on_dog & cotracker_visible[t]])
    dots[2].set_offsets(cotracker_tracks[t, on_dog & ~cotracker_visible[t]])
anim = FuncAnimation(fig, update, frames=T, interval=100)
plt.close()
HTML(anim.to_jshtml())